# 02 - Couche Silver : Transformation, Qualité et UDFs Python

Ce notebook lit les données Bronze (brutes), puis applique :
- Nettoyage : standardisation des noms de colonnes, cast des types, suppression des doublons
- UDFs Python : catégorisation énergétique et score carbone
- Qualité des données : contrôles qualité avec rapport détaillé
- Conformité RGPD : suppression des lignes exposant des données individuelles

Le résultat est écrit en Delta Lake Silver, prêt pour les agrégations Gold.

## Configuration et imports

In [0]:
# --- Imports Python ---

import unicodedata  # Module Python pour normaliser les caractères Unicode (ex : supprimer les accents)
import re           # Module Python pour les expressions régulières (nettoyage de chaînes de caractères)

from datetime import datetime  # Pour mesurer les durées et horodater les étapes

# --- Imports des fonctions PySpark ---

from pyspark.sql.functions import (
    col,               # Permet de référencer une colonne par son nom dans une expression Spark
    current_timestamp, # Retourne l'horodatage exact au moment de l'exécution du traitement
    current_date,      
    lit,               # Crée une colonne avec une valeur constante (même valeur pour toutes les lignes)
    when,              
    expr,              # Permet d'écrire une expression SQL sous forme de chaîne de caractères
    round as spark_round  # Arrondit une valeur numérique à N décimales (renommé pour éviter le conflit avec round() Python)
)

# --- Imports des types de données Spark ---

# Utilisés pour convertir (caster) les colonnes vers le bon type
from pyspark.sql.types import (
    DoubleType,   # Type virgule flottante double précision (pour les valeurs de consommation en MWh)
    IntegerType,  # Type entier (pour les années)
    StringType    # Type chaîne de caractères (pour les libellés, catégories...)
)

DATASET_NAME = "conso_inf36_region"

print("Configuration Silver chargée")

Configuration Silver chargée


## Chargement des données Bronze

In [0]:
# Lecture de la table Bronze depuis le catalogue Databricks

df_bronze = spark.read.table(f"bronze_{DATASET_NAME}")

print(f"Lignes chargees depuis Bronze : {df_bronze.count():,}")              

print(f"Colonnes disponibles : {df_bronze.columns}")

df_bronze.limit(3).display()

Lignes chargees depuis Bronze : 15,000
Colonnes disponibles : ['horodate', 'region', 'code_region', 'profil', 'plage_de_puissance_souscrite', 'nb_points_soutirage', 'total_energie_soutiree_wh', 'courbe_moyenne_ndeg1_wh', 'indice_representativite_courbe_ndeg1', 'courbe_moyenne_ndeg2_wh', 'indice_representativite_courbe_ndeg2', 'courbe_moyenne_ndeg1_ndeg2_wh', 'indice_representativite_courbe_ndeg1_ndeg2', 'jour_max_du_mois_0_1', 'semaine_max_du_mois_0_1', '_ingestion_timestamp', '_ingestion_date', '_source_dataset', '_source_url', '_source_description']


horodate,region,code_region,profil,plage_de_puissance_souscrite,nb_points_soutirage,total_energie_soutiree_wh,courbe_moyenne_ndeg1_wh,indice_representativite_courbe_ndeg1,courbe_moyenne_ndeg2_wh,indice_representativite_courbe_ndeg2,courbe_moyenne_ndeg1_ndeg2_wh,indice_representativite_courbe_ndeg1_ndeg2,jour_max_du_mois_0_1,semaine_max_du_mois_0_1,_ingestion_timestamp,_ingestion_date,_source_dataset,_source_url,_source_description
2025-06-30T22:00:00+00:00,Île-de-France,11,RES3,P1: ]0-9] kVA,73695,1.6409232E7,402.0,22,247.0,22,324.0,45,1,1,2026-04-29T15:39:00.117Z,2026-04-29,conso_inf36_region,https://opendata.enedis.fr/api/explore/v2.1/catalog/datasets/conso-inf36-region/exports/csv,Consommation résidentielle inférieure à 36 kVA par région (annuelle)
2024-09-30T22:00:00+00:00,Île-de-France,11,RES3,P1: ]0-9] kVA,73915,1.7178965E7,479.0,21,215.0,21,348.0,43,0,0,2026-04-29T15:39:00.117Z,2026-04-29,conso_inf36_region,https://opendata.enedis.fr/api/explore/v2.1/catalog/datasets/conso-inf36-region/exports/csv,Consommation résidentielle inférieure à 36 kVA par région (annuelle)
2024-03-31T22:00:00+00:00,Île-de-France,11,RES3,P1: ]0-9] kVA,61820,1.9123005E7,515.0,22,266.0,22,392.0,45,0,0,2026-04-29T15:39:00.117Z,2026-04-29,conso_inf36_region,https://opendata.enedis.fr/api/explore/v2.1/catalog/datasets/conso-inf36-region/exports/csv,Consommation résidentielle inférieure à 36 kVA par région (annuelle)


## UDFs Python - Catégorisation énergétique

Une UDF (User Defined Function) est une fonction personnalisée que l'on enregistre
dans Spark pour l'utiliser comme n'importe quelle fonction SQL sur les colonnes d'un DataFrame.

Note : dans un environnement de production avec un cluster Spark classique,
ces UDFs seraient implémentées en Scala pour de meilleures performances (Scala s'exécute
nativement sur la JVM de Spark, sans sérialisation/désérialisation Python).
Dans ce contexte Serverless, on utilisera l'équivalent Python qui produit le même résultat.

- Sérialiser : convertir chaque ligne du DataFrame du format JVM vers un format que Python comprend
- Désérialiser: reconvertir le résultat Python vers le format JVM pour continuer le traitement Spark

In [0]:
# Import nécessaire pour créer des UDFs PySpark
from pyspark.sql.functions import udf

from pyspark.sql.types import StringType, DoubleType            # Import du type de retour de l'UDF. StringType signifie que la fonction retourne une chaîne de caractères.


# --- UDF 1 : Catégorisation de la consommation électrique ---

# Définition de la fonction Python pure qui contient la logique métier
def categorize_consumption(consumption):

    if consumption is None:
        return "INCONNU"
    
    elif consumption < 1000.0:
        return "TRES FAIBLE"

    elif consumption < 5000.0:
        return "FAIBLE"

    elif consumption < 20000.0:
        return "MODEREE"

    elif consumption < 100000.0:
        return "ELEVEE"
    
    else:
        return "TRES ELEVEE"

categorize_consumption_udf = udf(categorize_consumption, StringType())

spark.udf.register("categorize_consumption_udf", categorize_consumption, StringType())

print("UDF categorize_consumption_udf enregistrée")

UDF categorize_consumption_udf enregistrée


In [0]:
# --- UDF 2 : Score carbone normalisé par site ---

def compute_carbon_score(consumption, nb_sites):

    if consumption is None or nb_sites is None or nb_sites == 0:
        return None

    conso_par_site = consumption / nb_sites                                # Calcul de la consommation moyenne par site (en MWh)

    score = min(100.0, round(conso_par_site / 50.0 * 10.0) / 10.0)          # Normalisation sur une echelle 0-100. On divise par 50 pour ramener a une échelle relative, on multiplie par 10 pour conserver une decimale apres l'arrondi, on divise par 10.0 pour retrouver la decimale, min(100.0, ...) plafonne le score a 100 maximum

    return score

compute_carbon_score_udf = udf(compute_carbon_score, DoubleType())
spark.udf.register("compute_carbon_score_udf", compute_carbon_score, DoubleType())

print("UDF compute_carbon_score_udf enregistree")

UDF compute_carbon_score_udf enregistree


## Nettoyage et standardisation des colonnes

In [0]:
def standardize_column_names(df):

    new_cols = []  # Liste qui va accumuler les nouveaux noms de colonnes

    for c in df.columns:

        # Étape 1 : décomposition Unicode NFD. NFD sépare les caractères et leurs accents en entités distinctes. Ex : "é" devient "e" + accent_aigu (deux entités séparées)
        
        normalized = unicodedata.normalize("NFD", c)

        # Étape 2 : encodage ASCII en ignorant les octets non-ASCII (les accents).


        ascii_str = normalized.encode("ascii", "ignore").decode("ascii")             # encode("ascii", "ignore") supprime tout ce qui n'est pas ASCII pur. decode("ascii") reconvertit les bytes en chaîne de caractères Python.

        # Étape 3 : remplacement de tous les caractères non-alphanumériques par "_".


        clean = re.sub(r"[^a-zA-Z0-9]", "_", ascii_str).lower()                  # [^a-zA-Z0-9] est une regex qui correspond à tout sauf les lettres et les chiffres. .lower() met le résultat en minuscules.

        # Étape 4 : nettoyage des underscores multiples ou en début/fin.

        clean = re.sub(r"_+", "_", clean).strip("_")                              # r"_+" correspond à un ou plusieurs underscores consécutifs et les remplace par un seul. .strip("_") supprime les underscores en début et en fin de chaîne.

        new_cols.append(clean)  # Ajout du nom nettoyé à la liste

    return df.toDF(*new_cols)                   # toDF() renomme toutes les colonnes du DataFrame en une seule opération. L'astérisque * déplie la liste en arguments séparés.


# --- Suppression des colonnes de métadonnées Bronze ---

# Ces colonnes ont été ajoutées par le notebook 01 pour la traçabilité Bronze. On les supprime ici car la couche Silver va créer ses propres métadonnées.

META_COLS = [
    "_ingestion_timestamp",
    "_ingestion_date",
    "_source_dataset",
    "_source_url",
    "_source_description"
]

# On ne supprime que les colonnes qui existent réellement dans le DataFrame (sécurité en cas d'évolution du schéma Bronze)

df_work = df_bronze.drop(*[c for c in META_COLS if c in df_bronze.columns])

# Application de la standardisation des noms de colonnes
df_work = standardize_column_names(df_work)

# --- Suppression des doublons ---
before_dedup = df_work.count()          # Comptage avant dédoublonnage
df_work      = df_work.dropDuplicates() # Supprime les lignes identiques sur toutes les colonnes
after_dedup  = df_work.count()          # Comptage après dédoublonnage

print(f"Doublons supprimés : {before_dedup - after_dedup:,}")

print("\nColonnes standardisées :")
for c in df_work.columns:
    print(f"  - {c}")

Doublons supprimés : 0

Colonnes standardisées :
  - horodate
  - region
  - code_region
  - profil
  - plage_de_puissance_souscrite
  - nb_points_soutirage
  - total_energie_soutiree_wh
  - courbe_moyenne_ndeg1_wh
  - indice_representativite_courbe_ndeg1
  - courbe_moyenne_ndeg2_wh
  - indice_representativite_courbe_ndeg2
  - courbe_moyenne_ndeg1_ndeg2_wh
  - indice_representativite_courbe_ndeg1_ndeg2
  - jour_max_du_mois_0_1
  - semaine_max_du_mois_0_1


## Controles qualité des données (Data Quality)

In [0]:
def run_data_quality_checks(df, dataset_name: str) -> list:

    checks     = []        # Accumulateur des résultats de tous les contrôles
    total_rows = df.count()

    print(f"\nCONTRÔLES QUALITÉ - {dataset_name.upper()}")
    print(f"Total lignes : {total_rows:,}")
    print("-" * 60)

    # --- Contrôle 1 : Valeurs nulles par colonne ---
    # Une colonne avec trop de valeurs nulles peut signaler un problème de source ou d'ingestion
    print("\nContrôle 1 : Valeurs nulles")

    for col_name in df.columns:

        null_count = df.filter(col(col_name).isNull()).count()  # Comptage des lignes où cette colonne est nulle

        null_pct = (null_count / total_rows * 100) if total_rows > 0 else 0  # Calcul du pourcentage de valeurs nulles par rapport au total de lignes

        # Classification du statut selon le seuil de valeurs nulles
        if null_pct == 0:
            status = "PASS"     # Aucune valeur nulle : parfait
        elif null_pct < 5:
            status = "WARNING"  # Quelques valeurs nulles : à surveiller
        else:
            status = "FAIL"     # Trop de valeurs nulles : problème probable

        # Enregistrement du résultat du contrôle
        checks.append({
            "check":      "null_check",
            "column":     col_name,
            "null_count": null_count,
            "null_pct":   round(null_pct, 2),
            "status":     status
        })

        # Affichage uniquement si des valeurs nulles sont détectées (évite de polluer les logs)
        if null_pct > 0:
            print(f"  [{status}] {col_name:<40} | {null_pct:5.1f}% nuls ({null_count:,})")


    # --- Contrôle 2 : Détection des doublons ---
    # Des doublons peuvent fausser les agrégations (sommes, moyennes) dans la couche Gold
    print("\nContrôle 2 : Doublons")
   
    dup_count  = total_rows - df.dropDuplicates().count()                    # La différence de count() donne le nombre exact de doublons.
    dup_status = "PASS" if dup_count == 0 else "WARNING"

    checks.append({
        "check":     "duplicate_check",
        "column":    "ALL",
        "dup_count": dup_count,
        "status":    dup_status
    })
    print(f"  [{dup_status}] Doublons détectés : {dup_count:,}")

    # --- Contrôle 3 : Valeurs négatives dans les colonnes numériques ---
    # Une consommation négative n'a pas de sens physique : cela signale une erreur de donnée
    print("\nContrôle 3 : Valeurs négatives (colonnes numériques)")

    # Filtrage des colonnes de type numérique dans le schéma Spark.
    numeric_cols = [
        f.name for f in df.schema.fields
        if str(f.dataType) in ("DoubleType()", "FloatType()", "LongType()", "IntegerType()")            # f.dataType est le type Spark de chaque champ, on le compare aux types numériques connus.
    ]

    for col_name in numeric_cols:
        
        # Comptage des lignes avec une valeur strictement négative pour cette colonne
        neg_count  = df.filter(col(col_name) < 0).count()
        neg_status = "PASS" if neg_count == 0 else "FAIL"

        checks.append({
            "check":     "negative_check",
            "column":    col_name,
            "neg_count": neg_count,
            "status":    neg_status
        })
        print(f"  [{neg_status}] {col_name:<40} | {neg_count:,} valeurs négatives")

    # Synthèse du rapport
    fails = sum(1 for c in checks if c["status"] == "FAIL")
    warns = sum(1 for c in checks if c["status"] == "WARNING")
    print(f"\nRésumé : {len(checks)} contrôles | {fails} FAIL | {warns} WARNING")

    return checks


# Exécution des contrôles qualité sur le DataFrame en cours de traitement
quality_report = run_data_quality_checks(df_work, DATASET_NAME)


CONTRÔLES QUALITÉ - CONSO_INF36_REGION
Total lignes : 15,000
------------------------------------------------------------

Contrôle 1 : Valeurs nulles
  [FAIL] total_energie_soutiree_wh                |   7.4% nuls (1,106)
  [FAIL] courbe_moyenne_ndeg1_wh                  |  18.2% nuls (2,727)
  [FAIL] courbe_moyenne_ndeg2_wh                  |  18.4% nuls (2,762)
  [FAIL] courbe_moyenne_ndeg1_ndeg2_wh            |  11.6% nuls (1,745)

Contrôle 2 : Doublons
  [PASS] Doublons détectés : 0

Contrôle 3 : Valeurs négatives (colonnes numériques)
  [PASS] code_region                              | 0 valeurs négatives
  [PASS] nb_points_soutirage                      | 0 valeurs négatives
  [PASS] total_energie_soutiree_wh                | 0 valeurs négatives
  [PASS] courbe_moyenne_ndeg1_wh                  | 0 valeurs négatives
  [PASS] courbe_moyenne_ndeg2_wh                  | 0 valeurs négatives
  [PASS] courbe_moyenne_ndeg1_ndeg2_wh            | 0 valeurs négatives
  [PASS] jour_max_du

## Détection dynamique des colonnes métier et cast des types

In [0]:
# --- Fonction de détection des colonnes par mots-clés ---


def detect_col(df, *keywords):
   
    for kw in keywords:
        match = next((c for c in df.columns if kw in c.lower()), None)
        if match:
            return match
    return None

# Assignation manuelle des colonnes métier d'après le vrai schéma Enedis.

conso_col  = "total_energie_soutiree_wh"   # Colonne de consommation électrique en Wh
annee_col  = None                          # Pas de colonne année directe : sera extraite depuis l'horodate
region_col = "region"                      # Colonne du libellé de région
sites_col  = "nb_points_soutirage"         # Colonne du nombre de points de soutirage

print("Colonnes métier assignées :")
print(f"  Consommation : {conso_col}")
print(f"  Année        : {annee_col}")
print(f"  Région       : {region_col}")
print(f"  Nb sites     : {sites_col}")

# --- Cast des types Spark ---

if conso_col:

    df_work = df_work.withColumn(conso_col, col(conso_col).cast(DoubleType()))          # cast(DoubleType()) convertit la colonne en nombre à virgule flottante.

if sites_col:
    df_work = df_work.withColumn(sites_col, col(sites_col).cast("long"))                # cast("long") convertit en Long (entier 64 bits) car le nombre de sites peut être grand

Colonnes métier assignées :
  Consommation : total_energie_soutiree_wh
  Année        : None
  Région       : region
  Nb sites     : nb_points_soutirage


## Application des UDFs Python

In [0]:
# --- UDF 1 : Categorisation de la consommation ---

if conso_col:
    df_work = df_work.withColumn(
        "categorie_consommation",                         # Nom de la nouvelle colonne creee
        expr(f"categorize_consumption_udf({conso_col})")  # Appel de l'UDF enregistree
    )
    print(f"UDF appliquee : categorie_consommation calculee depuis '{conso_col}'")

# --- UDF 2 : Score carbone normalise ---
# Cette UDF nécessite deux arguments : consommation et nombre de sites

if conso_col and sites_col:
    df_work = df_work.withColumn(
        "score_carbone_normalise",                                        # Nouvelle colonne de score
        expr(f"compute_carbon_score_udf({conso_col}, {sites_col})")       # Appel avec deux arguments
    )
    print(f"UDF appliquee : score_carbone_normalise calcule depuis '{conso_col}' et '{sites_col}'")

UDF appliquee : categorie_consommation calculee depuis 'total_energie_soutiree_wh'
UDF appliquee : score_carbone_normalise calcule depuis 'total_energie_soutiree_wh' et 'nb_points_soutirage'


## Conformité RGPD - k-anonymisation

Le RGPD interdit de publier des données permettant d'identifier indirectement
un individu. Dans le contexte Enedis, une ligne avec un très faible nombre de sites
dans une petite commune pourrait permettre d'identifier le profil de consommation
d'un foyer spécifique.

On applique la règle de k-anonymisation : toute ligne avec moins de k sites
est supprimée car elle pourrait exposer des données individuelles.

In [0]:
# Seuil de k-anonymisation : une ligne doit représenter au moins 5 sites différents pour être conservée. En dessous, le risque de ré-identification est trop élevé.

RGPD_MIN_SITES = 5

silver_pre_rgpd = df_work.count()           # Comptage avant le filtre pour mesurer l'impact

if sites_col:
    # On ne conserve que les lignes où le nombre de sites est connu ET supérieur au seuil.
    
    
    df_silver = df_work.filter(
        col(sites_col).isNotNull() & (col(sites_col) >= RGPD_MIN_SITES)             # col(sites_col).isNotNull() : on exclut les lignes sans information sur le nombre de sites. col(sites_col) >= RGPD_MIN_SITES : on exclut les lignes avec trop peu de sites.
    )

    silver_post_rgpd  = df_silver.count()
    lignes_supprimees = silver_pre_rgpd - silver_post_rgpd
    print(f"RGPD - Lignes supprimées (moins de {RGPD_MIN_SITES} sites) : {lignes_supprimees:,}")

else:
    # Si la colonne nb_sites n'est pas disponible, on ne peut pas appliquer le filtre. On journalise un avertissement mais on ne bloque pas le pipeline.
    df_silver = df_work
    print("Avertissement : colonne 'nb_sites' non trouvée, filtre RGPD non appliqué.")

# --- Ajout des métadonnées Silver ---
# Même principe que le notebook Bronze : on trace chaque étape de transformation.

df_silver = (
    df_silver
    .withColumn("_silver_timestamp", current_timestamp())  # Horodatage de la transformation Silver
    .withColumn("_silver_date",      current_date())       # Date de la transformation
    .withColumn("_rgpd_min_sites",   lit(RGPD_MIN_SITES))  # Seuil RGPD appliqué (pour l'audit)
)

print(f"Lignes Silver après nettoyage et RGPD : {df_silver.count():,}")

RGPD - Lignes supprimées (moins de 5 sites) : 0
Lignes Silver après nettoyage et RGPD : 15,000


## Écriture en Delta Lake Silver

In [0]:
table_name = f"silver_{DATASET_NAME}"

# Construction de l'objet writer Spark
writer = (
    df_silver.write                              # Accès à l'interface d'écriture du DataFrame
             .format("delta")                   # Format Delta Lake pour les transactions ACID et le versioning
             .mode("overwrite")                 # Remplace les données existantes si la table existe déjà
             .option("overwriteSchema", "true") # Accepte les changements de schéma
)

# Ajout du partitionnement si une colonne année a été détectée. Partitionner par année permet à Spark de ne lire que les partitions concernées lors des requêtes filtrées sur l'année (optimisation des performances).

if annee_col:
    writer = writer.partitionBy(annee_col)

writer.saveAsTable(table_name)

print(f"Données Silver écrites avec succès -> table : {table_name}")

# Aperçu des données Silver finales
df_silver.limit(5).display()

Données Silver écrites avec succès -> table : silver_conso_inf36_region


horodate,region,code_region,profil,plage_de_puissance_souscrite,nb_points_soutirage,total_energie_soutiree_wh,courbe_moyenne_ndeg1_wh,indice_representativite_courbe_ndeg1,courbe_moyenne_ndeg2_wh,indice_representativite_courbe_ndeg2,courbe_moyenne_ndeg1_ndeg2_wh,indice_representativite_courbe_ndeg1_ndeg2,jour_max_du_mois_0_1,semaine_max_du_mois_0_1,categorie_consommation,score_carbone_normalise,_silver_timestamp,_silver_date,_rgpd_min_sites
2023-12-31T23:00:00+00:00,Provence-Alpes-Côte d'Azur,93,PRO5,P4: ]9-12] kVA,2231,4170809.0,1828.0,41,1485.0,42,1653.0,84,0,0,TRES ELEVEE,37.4,2026-04-29T16:43:15.860Z,2026-04-29,5
2024-06-30T22:00:00+00:00,Provence-Alpes-Côte d'Azur,93,PRO5,P4: ]9-12] kVA,2122,6447151.0,918.0,43,1485.0,43,1200.0,87,0,0,TRES ELEVEE,60.8,2026-04-29T16:43:15.860Z,2026-04-29,5
2023-12-31T23:00:00+00:00,Provence-Alpes-Côte d'Azur,93,PRO5,P6: ]15-18] kVA,1055,2435911.0,2038.0,39,2057.0,42,2048.0,81,0,0,TRES ELEVEE,46.2,2026-04-29T16:43:15.860Z,2026-04-29,5
2025-06-30T22:00:00+00:00,Provence-Alpes-Côte d'Azur,93,PRO5,P6: ]15-18] kVA,962,3857827.0,2530.0,43,2342.0,44,2436.0,87,1,1,TRES ELEVEE,80.2,2026-04-29T16:43:15.860Z,2026-04-29,5
2023-09-30T22:00:00+00:00,Provence-Alpes-Côte d'Azur,93,RES1 (+ RES1WE),P1: ]0-3] kVA,113708,4364431.0,68.0,4,55.0,4,53.0,8,0,0,TRES ELEVEE,0.8,2026-04-29T16:43:15.860Z,2026-04-29,5


## Recapitulatif de la couche Silver

| Etape | Action realisee |
|---|---|
| Nettoyage | Standardisation des noms de colonnes, suppression des doublons |
| Cast des types | Consommation en Double, annee en Integer, nb_sites en Long |
| Data Quality | Controles : nulls, doublons, valeurs negatives |
| UDF Scala | categorize_consumption_scala et compute_carbon_score_scala |
| RGPD | k-anonymisation : suppression des lignes avec moins de 5 sites |
| Stockage | Delta Lake Silver, partitionne par annee |

Nous arrivons ainsi à la fin du step Silver. Passeons au notebook 03_Gold_Aggregations pour produire les KPIs metier.